# 33 — Old zone beside new zone: accuracy by the edge that decides the call

All the logic lives in `src/accuracy_by_edge.py`.

**Ground rule.** Every season is graded against its own zone: the old zone for 2021–2025, the new ABS zone for 2026. Nothing is re-graded, re-weighted or standardised.

**The idea.** The side edges of the zone are defined the same way under both zones: the 17-inch plate plus the ball. The top and the bottom are where the definition changed in 2026. So we split called pitches by the edge that decides the call and look at each group over time.

**Regions** (from `grade_pitches.py`): thirds of each season's own zone by height (low, mid, high) and thirds of the plate by width (inside, middle, outside). `mid-inside` and `mid-outside` are decided only by a side edge; `high-middle` only by the top; `low-middle` only by the bottom; the four `high-`/`low-` `inside`/`outside` regions by both. `corner` is off the plate and outside the zone's height; `mid-middle` is the heart of the zone.

In [ ]:
import sys

import pandas as pd
from IPython.display import display

sys.path.insert(0, "../src")
import accuracy_by_edge

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

tables = accuracy_by_edge.build()
accuracy_by_edge.save(tables)

by_season = tables["accuracy_by_edge_season"]
change = tables["accuracy_by_edge_change"]
shares = tables["region_shares_by_season"]

## 1. The picture: three edge families

In [ ]:
figures = accuracy_by_edge.make_figures(tables)

## 2. Accuracy by season (%), at three levels of detail

In [ ]:
def accuracy_table(level):
    rows = by_season[by_season["level"] == level]
    table = rows.pivot(index=["rule", "name"], columns="season", values="accuracy")
    return (table * 100).round(2)


for level in ("edge_family", "edge_group", "zone_region"):
    print(level)
    display(accuracy_table(level))

## 3. The 2026 change beside the trend before it

`yearly_change_2021_2025_pp` is the average change per year under the old zone. `change_2025_2026_pp` is the step into the new zone, with a 95% interval from resampling whole games. `break_from_trend_pp` is the difference between the two.

In [ ]:
columns = [
    "level",
    "name",
    "rule",
    "yearly_change_2021_2025_pp",
    "change_2025_2026_pp",
    "change_low_pp",
    "change_high_pp",
    "break_from_trend_pp",
]
change[columns].round(2)

## 4. Where the called pitches are, as recorded (% of called pitches)

No re-weighting: this only shows how the mix of calls differs under the two zones.

In [ ]:
(shares.set_index("zone_region") * 100).round(1)

## Takeaways

1. **Side edges, same rule in both zones:** 89.3% correct in 2021, 91.8% in 2025, 92.6% in 2026. They gained about 0.6 pp a year under the old zone and 0.9 pp in 2026 (95% interval 0.5 to 1.2). Where the rule did not change, 2026 continued the earlier pace.
2. **Top of the zone, rule changed:** 89.1% in 2021 and 89.2% in 2025, flat for five seasons, then 90.9% in 2026: +1.7 pp (1.4 to 2.0). This is the break from trend. It sits in the two top corners (+2.1 pp together); the top-middle moved +0.4 pp, which is within noise.
3. **Bottom of the zone, rule changed:** 90.9% in 2021, 92.1% in 2025, 91.7% in 2026: −0.4 pp (−0.6 to −0.1). The bottom-middle alone lost 1.4 pp (−1.8 to −1.0).
4. **No edge involved:** the heart of the zone (99.9%) and pitches outside on both axes (98.1%) moved by less than 0.1 pp.
5. **The mix of calls differs under the new zone.** The new zone is shorter, so more off-the-plate pitches also miss on height: the easy `corner` region holds 25.6% of called pitches in 2026 against 22.9% in 2025. We show this; we do not adjust for it.
6. **What this says and does not say.** It shows where the 2026 change sits: at the top, where the zone itself changed. It does not say whether umpires adjusted to the new top or the new top is simply closer to what they already called; Jorge's called-zone table (J9) speaks to that.

Outputs: `accuracy_by_edge_season.csv`, `accuracy_by_edge_change.csv`, `region_shares_by_season.csv` in `reports/tables`; `accuracy_by_edge.png` in `reports/figures`.